# 03. 윈도우 → 문장 변환

**목적**: 재영 윈도우 파일의 `signature`(기호 덩어리)를 모델이 읽을 영어 문장으로 바꾸고, 확정 라벨과 함께 `windows_A.jsonl`, `windows_B.jsonl`로 저장한다. 이 파일이 다음 단계(04. 분할)의 입력이다.

**바꾸기 전**
```
ssm.amazonaws.com|DescribeInstanceInformation||True|IAMUser|False|50|actor0 > ssm.amazonaws.com|SendCommand||False|IAMUser|False|1|actor0 > ...
```
**바꾼 후**
```
CloudTrail window: 4 call groups by 1 actor (IAMUser), no failures, region unchanged. Sequence: ssm DescribeInstanceInformation (read, x50) -> ssm SendCommand (write) -> ...
```

**문장 규칙**
- 순서는 그대로 유지 (`->`로 연결). 공격 흐름이 순서에 담겨 있기 때문
- 서비스는 짧게 (`ssm.amazonaws.com` → `ssm`)
- 읽기/쓰기, 반복 횟수(2번 이상일 때 `xN`), 실패 시 에러 코드
- 행위자가 바뀌면 `[actor2]`처럼 표시
- 넣지 않는 것: IP, userAgent, 사용자 이름, 시나리오 이름 (정답 누수)

**준비**: 02와 같음 (드라이브에 zip, labels.csv). CPU로 충분, 1~2분.

## 0. 파일 찾기, 압축 풀기 (02와 같음)

In [ ]:
import os, zipfile, glob, json
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    ROOT = '/content/drive/MyDrive'
    WORK = '/content/data'
except ImportError:
    ROOT = os.environ.get('DATA_DIR', '.')
    WORK = os.path.join(ROOT, 'unzipped')

zip_hits = glob.glob(f'{ROOT}/**/windows_B_size5_stride1*.zip', recursive=True)
lab_hits = glob.glob(f'{ROOT}/**/labels.csv', recursive=True)
assert zip_hits and lab_hits, 'zip 또는 labels.csv를 못 찾음'
DATA_DIR = os.path.dirname(lab_hits[0])
os.makedirs(WORK, exist_ok=True)
zipfile.ZipFile(zip_hits[0]).extractall(WORK)
print('DATA_DIR =', DATA_DIR, '| WORK =', WORK)

## 1. 불러오기, 확정 라벨 적용 (02의 7번과 같은 규칙)

[하는 일] 원래 전술 → 우리 라벨, 제외 라벨 윈도우 삭제, 같은 문장은 하나만, 우리 라벨 기준 충돌 문장은 삭제.

[볼 것] A 5,199개, B 4,282개가 나오면 02와 같은 결과.

In [ ]:
import pandas as pd, numpy as np
pd.set_option('display.max_colwidth', 200)

wins = {k: pd.read_csv(f'{WORK}/windows_{k}_size5_stride1.csv', low_memory=False) for k in 'AB'}
labels = pd.read_csv(lab_hits[0])
to_label = {t: r.label_id for r in labels.itertuples() for t in str(r.source_tactics).split('|') if t and t != 'nan'}
role = dict(zip(labels.label_id, labels.role))

def apply_labels(k):
    u = wins[k].copy()
    tac = u.m1_tactics.where(u.m1_is_attack.astype(bool), '').fillna('')
    u['labels'] = tac.apply(lambda s: sorted({to_label[t] for t in s.split('|') if t}) or ['BENIGN'])
    u = u[~u['labels'].apply(lambda ls: any(role[l] == 'excluded' for l in ls))]
    u['key'] = u['labels'].apply('|'.join)
    n_lab = u.groupby('signature').key.nunique()
    u = u[~u.signature.isin(n_lab[n_lab > 1].index)].drop_duplicates('signature')
    return u.reset_index(drop=True)

final = {k: apply_labels(k) for k in 'AB'}
print({k: len(v) for k, v in final.items()})

## 2. 문장 변환 함수

[하는 일] signature 한 개를 영어 문장 한 개로 바꾼다.

signature 안 이벤트 하나 = `서비스 | API | 에러 | 읽기전용 | 사용자유형 | 리전변경 | 반복횟수 | 행위자`

[확인 질문] 문장에 순서가 남아 있나? 반복 횟수가 보이나?

In [ ]:
def parse(sig):
    out = []
    for e in sig.split(' > '):
        src, name, err, ro, utype, reg, rep, actor = e.split('|')
        out.append(dict(svc=src.split('.')[0], name=name, err=err, read=(ro == 'True'),
                        utype=utype or 'unknown', region=(reg == 'True'), rep=int(float(rep or 1)), actor=actor))
    return out

def to_text(sig):
    ev = parse(sig)
    actors = list(dict.fromkeys(e['actor'] for e in ev))
    utypes = list(dict.fromkeys(e['utype'] for e in ev))
    fails = [e for e in ev if e['err']]
    head = (f"CloudTrail window: {len(ev)} call group{'s' if len(ev) > 1 else ''} by {len(actors)} actor"
            f"{'s' if len(actors) > 1 else ''} ({', '.join(utypes)}), "
            f"{'%d failed' % len(fails) if fails else 'no failures'}, "
            f"region {'changed' if any(e['region'] for e in ev) else 'unchanged'}.")
    steps, prev = [], None
    for e in ev:
        tags = ['read' if e['read'] else 'write']
        if e['rep'] > 1: tags.append(f"x{e['rep']}")
        if e['err']: tags.append(f"failed: {e['err']}")
        who = f"[{e['actor']}] " if prev is not None and e['actor'] != prev else ''
        steps.append(f"{who}{e['svc']} {e['name']} ({', '.join(tags)})")
        prev = e['actor']
    return head + ' Sequence: ' + ' -> '.join(steps)

ex = final['B'].sort_values('n_events').iloc[0]
print('바꾸기 전:', ex.signature[:300])
print('\n바꾼 후 :', to_text(ex.signature))

## 3. 전체 변환 + 예시 보기

[하는 일] A, B 전체 윈도우를 문장으로 바꾸고 라벨별로 하나씩 본다.

문장은 리전 변경을 "어느 이벤트에서" 바뀌었는지까지는 안 적고 윈도우 전체로 한 번만 적는다. 그래서 signature는 달라도 문장은 같아지는 경우가 생기는데 (A 약 200개, B 약 50개), 라벨이 같으면 하나로 합친다. 02의 중복 합치기와 같은 원리.

[볼 것] 같은 라벨끼리 문장이 비슷한지, 다른 라벨과는 다른지.

[확인 질문] 문장만 보고 라벨을 짐작할 수 있나? (사람이 짐작 가능하면 모델도 배울 여지가 있음)

In [ ]:
for k in 'AB':
    f = final[k]
    f['text'] = f.signature.apply(to_text)
    n_lab = f.groupby('text').key.nunique()
    bad = n_lab[n_lab > 1].index                       # 문장은 같은데 라벨이 다르면 삭제
    before = len(f)
    final[k] = f[~f.text.isin(bad)].drop_duplicates('text').reset_index(drop=True)
    print(f'[{k}] {before} -> {len(final[k])} (같은 문장 합침, 라벨 충돌 문장 {len(bad)}개 삭제)')
print()

name = dict(zip(labels.label_id, labels.name))
for lid in labels[labels.role != 'excluded'].label_id:
    hit = final['B'][final['B']['labels'].apply(lambda ls: lid in ls)]
    if len(hit):
        print(f'■ {name[lid]} ({lid}) - B {len(hit)}개 중 예시')
        print('  ', hit.text.iloc[len(hit) // 2][:400], '\n')

## 4. 문장 길이 확인 (토큰)

[하는 일] edge 토크나이저로 문장 길이를 잰다. 라벨 문장(약 800토큰)과 합쳐도 한계 7,999 안에 들어가야 한다.

[볼 것] 최대 길이가 수백 토큰 수준이면 충분히 여유.

In [ ]:
try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained('knowledgator/gliclass-edge-v3.0')
    lab_tok = sum(len(tok(t, add_special_tokens=False)['input_ids']) + 1 for t in labels[labels.role != 'excluded'].label_text)
    for k in 'AB':
        n = final[k].text.apply(lambda t: len(tok(t)['input_ids']))
        print(f'[{k}] 윈도우 문장 토큰: 평균 {n.mean():.0f} / 최대 {n.max()} | 라벨 8개 문장 {lab_tok} | 합 최대 {n.max() + lab_tok} / 7999')
except Exception as e:
    print('토크나이저를 못 불러옴 (인터넷 확인):', e)

## 5. 저장

[하는 일] 다음 단계(분할)에서 쓸 파일로 저장한다. 데이터라서 저장소가 아니라 드라이브에 둔다.

저장 칸: `text`(입력 문장), `labels`(정답 label_id 리스트), `session`, `window`(세션 안 순서), `source`, `n_attack_events`

In [ ]:
OUT = f'{DATA_DIR}/processed'
os.makedirs(OUT, exist_ok=True)
cols = ['text', 'labels', 'session', 'window', 'source', 'n_attack_events']
for k in 'AB':
    path = f'{OUT}/windows_{k}.jsonl'
    final[k][cols].to_json(path, orient='records', lines=True, force_ascii=False)
    print('저장:', path, len(final[k]), '줄')
print(open(f'{OUT}/windows_B.jsonl').readline()[:400])